# 07. The drone, modular: MCDPL-style operator syntax

The same MCDP as notebooks **01** and **06**, but with battery and actuator defined as **independent** subsystems and assembled with the `System` builder. This notebook uses the new operator-overloaded constraint syntax: each connection is written as a Python `>=` between port handles, mirroring how the same model would appear in the MCDPL source from the paper.

This is the recommended style for modular design. Each subsystem is a `Module` subclass with its own `F`, `R`, and `h`. The wiring at the bottom reads as a column of inequalities, the way a textbook would.


## Imports

In [1]:
from codesign import Module, Reals, System, solve

## Subsystems as class-based Modules

Each is a self-contained design problem. The constructor accepts parameters so a single class can be reused with different physical constants.


In [2]:
class Battery(Module):
    F = {"capacity": Reals(unit="J")}
    R = {"mass":     Reals(unit="kg")}

    def __init__(self, specific_energy=1.8e6):
        self.specific_energy = specific_energy
        super().__init__()

    def h(self, f):
        return {"mass": f["capacity"] / self.specific_energy}


class Actuator(Module):
    F = {"lift_force": Reals(unit="N")}
    R = {"power":      Reals(unit="W")}

    def __init__(self, c_lift=10.0):
        self.c_lift = c_lift
        super().__init__()

    def h(self, f):
        return {"power": self.c_lift * f["lift_force"] ** 2}


Battery(), Actuator()

(DP(battery: capacity:R+[J] -> A[mass:R+[kg]]),
 DP(actuator: lift_force:R+[N] -> A[power:R+[W]]))

## Assembly via operator-overloaded constraints

`sys.provides`, `sys.requires`, and `sys.add` each return a port handle. Arithmetic operators on handles build expression trees lazily; the `>=` operator registers a constraint with the parent system.


In [3]:
G = 9.81

sys = System("drone")

# Outer interface: each call returns a Port handle.
endurance     = sys.provides("endurance",     unit="s")
extra_payload = sys.provides("extra_payload", unit="kg")
extra_power   = sys.provides("extra_power",   unit="W")
total_mass    = sys.requires("total_mass",    unit="kg")

# Subsystems: each call returns a ModuleHandle whose attributes are ports.
battery  = sys.add("battery",  Battery())
actuator = sys.add("actuator", Actuator())

# Connection constraints. Read like a textbook page of inequalities.
battery.capacity    >= (actuator.power + extra_power) * endurance
actuator.lift_force >= G * (battery.mass + extra_payload)
total_mass          >= battery.mass + extra_payload

print(sys)

System('drone'):
  provides:
    endurance: R+[s]
    extra_payload: R+[kg]
    extra_power: R+[W]
  requires:
    total_mass: R+[kg]
  subsystems:
    battery: (capacity) -> (mass)
    actuator: (lift_force) -> (power)
  constraints:
    battery.capacity >= ((actuator.power + extra_power) * endurance)
    actuator.lift_force >= (9.81 * (battery.mass + extra_payload))
    total_mass >= (battery.mass + extra_payload)


## Build and solve

In [4]:
drone = sys.build()
print(drone)
print()

cases = [
    ("Short, light",   dict(endurance=60.0,   extra_payload=0.10, extra_power=1.0)),
    ("Medium, modest", dict(endurance=300.0,  extra_payload=0.50, extra_power=5.0)),
    ("Longer mission", dict(endurance=600.0,  extra_payload=0.50, extra_power=5.0)),
    ("Marginal",       dict(endurance=600.0,  extra_payload=1.00, extra_power=10.0)),
    ("Infeasible",     dict(endurance=1800.0, extra_payload=1.00, extra_power=10.0)),
]
for label, f in cases:
    result = solve(drone, f, max_iter=200)
    print(f"{label:<16} iters={result.iterations:>3}  "
          f"feasible={result.feasible}  {result.antichain}")

DP(loop(drone_inner, axis=__modules__): endurance:R+[s]×extra_payload:R+[kg]×extra_power:R+[W] -> A[total_mass:R+[kg]])

Short, light     iters= 17  feasible=True  Antichain[(total_mass=0.1004 kg)]
Medium, modest   iters= 43  feasible=True  Antichain[(total_mass=0.5492 kg)]
Longer mission   iters= 81  feasible=True  Antichain[(total_mass=0.6283 kg)]
Marginal         iters= 32  feasible=False  Antichain[(total_mass=⊤)]
Infeasible       iters= 16  feasible=False  Antichain[(total_mass=⊤)]


## What changed compared to notebook 01

The values are identical (for Medium, modest: `total_mass = 0.5492 kg = 0.04921 (battery) + 0.5 (payload)`). The Kleene loop now updates each subsystem's R ports in alternation, so the iteration count is somewhat larger than the monolithic version. The fixed point is the same.

The payoff is **clarity and modularity**: `Battery` and `Actuator` are reusable building blocks, the constraints between them read as math rather than dict lookups, and adding a third subsystem (notebook **08**) or building a heterogeneous network (notebook **09**) requires no extra machinery.

### The same example in the lambda-based legacy syntax

For comparison, here is what the constraint block looked like before:

```python
sys.constrain("battery.capacity",
              lambda x: (x["actuator.power"] + x["extra_power"]) * x["endurance"])
sys.constrain("actuator.lift_force",
              lambda x: G * (x["battery.mass"] + x["extra_payload"]))
sys.constrain("total_mass",
              lambda x: x["battery.mass"] + x["extra_payload"])
```

Both forms still work and compile to the same internal constraint list. The operator form is what we recommend for new code.


## System structure with `viz.to_dot`

The constraint graph is implicit in the operator overloading; `viz.to_dot` extracts it as a GraphViz dot string. Render it externally with `dot -Tpng` or paste into [graphviz online](https://dreampuf.github.io/GraphvizOnline/) to see modules, outer ports, and the edges connecting them.


In [5]:
from codesign import viz
dot = viz.to_dot(drone, name="drone")
print(dot)

digraph drone {
  rankdir=LR;
  node [shape=box, style="rounded,filled", fillcolor="#f6f6f6", fontname="Helvetica"];
  edge [fontname="Helvetica", fontsize=10];
  subgraph cluster_sys1 {
    label="System"; style="rounded,dashed"; color="#888888";
    mod2 [label="battery\nF: capacity\nR: mass", fillcolor="#e8f0fe"];
    mod3 [label="actuator\nF: lift_force\nR: power", fillcolor="#e8f0fe"];
    mod3 -> mod2 [label="((actuator.power + extra_power)...", color="#555"];
    mod2 -> mod3 [label="(9.81 * (battery.mass + extra_p...", color="#555"];
  }
}
